In [ ]:
import torch
import triton
import triton.language as tl

@triton.jit                               
def add_kernel(x_ptr, y_ptr, z_ptr,
               n_elements,
               BLOCK_SIZE: tl.constexpr): 
    pid = tl.program_id(0)                
    block_start = pid * BLOCK_SIZE
    offsets = block_start + tl.arange(0,
                                     BLOCK_SIZE)   
    mask = offsets < n_elements

    x = tl.load(x_ptr + offsets,
                mask=mask)                
    y = tl.load(y_ptr + offsets,
                mask=mask)                
    tl.store(z_ptr + offsets,
             x + y,
             mask=mask)                   

def add(x: torch.Tensor,
        y: torch.Tensor,
        BLOCK_SIZE: int = 1024):          
    assert x.is_cuda and y.is_cuda, \
        "Inputs must be on CUDA device"
    assert x.shape == y.shape, \
        "Input shapes must match"

    z = torch.empty_like(x)
    n_elements = x.numel()

    grid = lambda meta: (                 
        triton.cdiv(n_elements,
                    meta['BLOCK_SIZE']),
    )

    add_kernel[grid](x, y, z,
                     n_elements,
                     BLOCK_SIZE=BLOCK_SIZE) 
    return z

# Execution and Validation
torch.manual_seed(0)
size = 98432
x = torch.rand(size,
               device="cuda",
               dtype=torch.float32)
y = torch.rand(size,
               device="cuda",
               dtype=torch.float32)

output_torch = x + y
output_triton = add(x, y)

max_diff = torch.max(
    torch.abs(output_torch - output_triton)
).item()

print(f"Maximum absolute difference: {max_diff:.3e}")
assert torch.allclose(output_triton, output_torch)
print("OK — Triton result matches PyTorch.")

In [ ]:
import os

vector_addition = add # Alias to match the benchmark script referencing

@triton.testing.perf_report(
    triton.testing.Benchmark(
        x_names=['size'],
        x_vals=[2**i for i in range(10, 25)],
        x_log=True,
        line_arg='provider',
        line_vals=['python', 'torch', 'triton'],
        line_names=['Python (CPU loop)', 'PyTorch (GPU)', 'Triton (GPU)'],
        styles=[('red', '-'), ('blue', '-'), ('green', '-')],
        ylabel='GB/s',
        plot_name='vector_addition_performance',
        args={},
    )
)
def benchmark(size, provider):
    quantiles = [0.5, 0.2, 0.8]
    x_gpu = torch.rand(size, device='cuda', dtype=torch.float32)
    y_gpu = torch.rand_like(x_gpu)
    if provider == 'python':
        x_cpu = x_gpu.cpu().tolist()
        y_cpu = y_gpu.cpu().tolist()
        def python_add():
            return [a + b for a, b in zip(x_cpu, y_cpu)]
        ms, min_ms, max_ms = triton.testing.do_bench(python_add, quantiles=quantiles)
    elif provider == 'torch':
        ms, min_ms, max_ms = triton.testing.do_bench(lambda: x_gpu + y_gpu, quantiles=quantiles)
    elif provider == 'triton':
        ms, min_ms, max_ms = triton.testing.do_bench(lambda: vector_addition(x_gpu, y_gpu), quantiles=quantiles)
    gbps = lambda ms: 12 * size / (ms * 1e-3) / 1e9
    return gbps(ms), gbps(min_ms), gbps(max_ms)

# Create the directory if it doesn't exist
output_path = './vec_add_perf'
os.makedirs(output_path, exist_ok=True)

# Run the benchmark suite
benchmark.run(print_data=True, show_plots=True, save_path=output_path)

In [ ]:
import torch
import triton
import triton.language as tl

@triton.jit
def add_kernel_ir(x_ptr, y_ptr, out_ptr, n, BLOCK: tl.constexpr):
    pid = tl.program_id(0)
    offs = pid * BLOCK + tl.arange(0, BLOCK)
    mask = offs < n
    x = tl.load(x_ptr + offs, mask=mask)
    y = tl.load(y_ptr + offs, mask=mask)
    tl.store(out_ptr + offs, x + y, mask=mask)

x_test = torch.rand(1024, device="cuda")
y_test = torch.rand(1024, device="cuda")
out_test = torch.empty_like(x_test)
compiled = add_kernel_ir[(1,)](x_test, y_test, out_test, 1024, BLOCK=1024)

with open("ttir.md", "w") as f:
    f.write(compiled.asm["ttir"])

with open("ttgir.md", "w") as f:
    f.write(compiled.asm["ttgir"])

with open("llir.md", "w") as f:
    f.write(compiled.asm["llir"])

with open("ptx.md", "w") as f:
    f.write(compiled.asm["ptx"])

print("Triton generated IR files successfully saved.")